# Mạng nơ-ron với PyTorch: phân loại hoa Iris

*Bài 9, mục 9.1.3.* Ví dụ theo giáo trình (Chương 13). PyTorch là thư viện học sâu phổ biến: tự động tính gradient bằng lan truyền ngược (`loss.backward()`), hỗ trợ GPU. Cài đặt: `pip install torch` (xem https://pytorch.org).

Kiến trúc: 4 đầu vào → lớp ẩn 10 nơ-ron ReLU → lớp ẩn 8 nơ-ron ReLU → 3 đầu ra.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(0)                         # cố định khởi tạo ngẫu nhiên để kết quả lặp lại được
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [2]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 10)          # lớp ẩn thứ nhất: z = W x + b
        self.fc2 = nn.Linear(10, 8)          # lớp ẩn thứ hai
        self.fc3 = nn.Linear(8, 3)           # lớp ra: 3 điểm số (softmax nằm trong hàm mất mát)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)


model = NeuralNetwork()
print(model)
print('Số tham số:', sum(p.numel() for p in model.parameters()))

NeuralNetwork(
  (fc1): Linear(in_features=4, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=8, bias=True)
  (fc3): Linear(in_features=8, out_features=3, bias=True)
  (relu): ReLU()
)
Số tham số: 165


`CrossEntropyLoss` gộp softmax và cross-entropy. Mỗi epoch: lan truyền xuôi, tính mất mát, `zero_grad` xóa gradient cũ, `backward` lan truyền ngược, `step` cập nhật trọng số (thuật toán Adam).

In [3]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
for epoch in range(100):
    outputs = model(X_train)                 # lan truyền xuôi
    loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    loss.backward()                          # lan truyền ngược
    optimizer.step()                         # cập nhật trọng số
    if (epoch + 1) % 20 == 0:
        print('Epoch %3d: loss = %.4f' % (epoch + 1, loss.item()))

Epoch  20: loss = 0.6268
Epoch  40: loss = 0.3895
Epoch  60: loss = 0.2425
Epoch  80: loss = 0.1169
Epoch 100: loss = 0.0614


In [4]:
with torch.no_grad():
    predicted = model(X_test).argmax(dim=1)
accuracy = (predicted == y_test).float().mean().item()
print('Độ chính xác trên tập kiểm tra: %.4f' % accuracy)

Độ chính xác trên tập kiểm tra: 1.0000
